In [ ]:
# customer reviews dataset with category count

import pandas as pd

df = pd.read_csv("customer_reviews.csv")

print(df.shape)
print(df.head())
print(df["category"].value_counts())

In [ ]:
# Displaying table of customer reviews

df = pd.read_csv("customer_reviews.csv")
df.head(10)

In [ ]:
# Cleaning the customer reviews data
import spacy

nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    doc = nlp(text)
    tokens = [
        token.lemma_.lower() 
        for token in doc 
        if token.is_alpha and not token.is_stop ]
   
    return " ".join(tokens)

df["clean_review"] = df["review"].apply(clean_text)

In [ ]:
# Top recurring terms in each issue category (identifying what matters most to customers).

from collections import Counter

categories = df["category"].unique()

for cat in categories:
    subset = df[df["category"] == cat]
    all_text = " ".join(subset["review"])
    common = Counter(all_text.split()).most_common(15)
    
    print(f"Top terms for category: {cat}")
    print(common)
    print("\n")

In [ ]:
# counting how many reviews fall into each category

summary = df["category"].value_counts().reset_index()
summary.columns = ["Category", "Count"]
print(summary)

In [ ]:
# Bar Chart of which category is most frequent and important to address.

import matplotlib.pyplot as plt

plt.bar(summary["Category"], summary["Count"])
plt.xticks(rotation=45)
plt.title("Category Distribution")
plt.show()

In [ ]:
# Bar Chart of frequent customer complaint keywords

import spacy
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

# Loading SpaCy ( AI Tool)
nlp = spacy.load("en_core_web_sm")

# Loading customer review dataset
df = pd.read_csv("customer_reviews.csv")

# Processing text
tokens = []
for doc in df["review"].dropna():
    processed = nlp(doc.lower())
    tokens.extend([
        token.lemma_ for token in processed
        if token.is_alpha and not token.is_stop ])

# Counting most common keywords
word_freq = Counter(tokens)
common_words = word_freq.most_common(10)

# Converting to DataFrame
freq_df = pd.DataFrame(common_words, columns=["Keyword", "Frequency"])

# Plotting word frequency
freq_df.plot(kind="bar", x="Keyword", y="Frequency", legend=False)
plt.title("Most Frequent Customer Complaint Keywords")
plt.xlabel("Keyword")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [1]:
# necessary libraries imported for my NASA Dataset

import numpy as np
import os
from scipy import signal
import matplotlib.pyplot as plt
import cv2

In [ ]:
# Loading and displaying one of the vibration signal files

folder = r"C:\Users\lenovo\Bearings_data\1st_test\1st_test" 
filename = "2003.10.22.12.06.24"  
file_path = os.path.join(folder, filename)

# Loading the data 
data = np.loadtxt(file_path, delimiter='\t')  

# Take channel 1 (usually the drive-end sensor—check your dataset docs if needed)
vibration = data[:, 0]  # 1D array of ~20480 points

print("Signal loaded. Length:", len(vibration))
print("First 10 values:", vibration[:10])

In [ ]:
# Generating spectrogram image

fs = 20000  # Hz

# Computing Short-Time Fourier Transform (STFT)
f, t, Sxx = signal.spectrogram(vibration, fs=fs, nperseg=256, noverlap=128, nfft=512)

# Converting to dB for better visuals
Sxx_db = 10 * np.log10(Sxx + 1e-10)  # Avoid log of zero

# Normalizing to image range (0-255)
Sxx_norm = cv2.normalize(Sxx_db, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Adding color  for better visual
img = cv2.applyColorMap(Sxx_norm, cv2.COLORMAP_JET)

# Resizing the model
img = cv2.resize(img, (224, 224))

# Showing the output
cv2.imwrite("spectrogram_example.jpg", img)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # Fix color for plot
plt.title("Spectrogram from Bearing Signal")
plt.axis('off')
plt.show()

In [ ]:
# Loading new spectrogram image for better visuals and to count edges for defect severity (for one file)

spectro_img = cv2.imread("spectrogram_example.jpg", cv2.IMREAD_GRAYSCALE)  

# Edge detection 
edges = cv2.Canny(spectro_img, 100, 200)  # Thresholds—tweak as needed

# Showing the output
plt.imshow(edges, cmap='gray')
plt.title("Edges in Spectrogram (Potential Defect Indicators)")
plt.axis('off')
plt.show()

# Measuring the edge count for variability
edge_count = np.sum(edges > 0)
print("Number of edge pixels (rough variability measure):", edge_count)

In [ ]:
# Generating Gray spectrogram images for all files

import cv2
import os

input_folder  = r"C:\Users\lenovo\Bearings_data\1st_test\1st_test\spectrograms"
output_folder = os.path.join(input_folder, "processed_edges")   

os.makedirs(output_folder, exist_ok=True)

# Getting list of all JPG files
image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Found {len(image_files)} images. Processing...")

for i, filename in enumerate(image_files, 1):
    input_path = os.path.join(input_folder, filename)
    
    try:
        img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            print(f"Skipped {filename} — couldn't load")
            continue
        
        # Edge detection
        edges = cv2.Canny(img, 80, 200)   
        
        # Saving result
        output_path = os.path.join(output_folder, f"edges_{filename}")
        cv2.imwrite(output_path, edges)
        
        print(f"[{i}/{len(image_files)}] Processed and saved: {filename}")
        
    except Exception as e:
        print(f"Error on {filename}: {e}")

print(f"\nAll done. Results saved in: {output_folder}")

In [ ]:
# Calculating RMS and Laplacian Variance and saved as a CSV file

import numpy as np
import os
from scipy import signal
import cv2

folder = r"C:\Users\lenovo\Bearings_data\1st_test\1st_test"
file_limit = 100 

file_names = []
rms_list = []
variance_list = []

for i, filename in enumerate(os.listdir(folder)):
    if not filename.startswith('2003'): continue
    if i >= file_limit: break

    file_path = os.path.join(folder, filename)
    try:
        data = np.loadtxt(file_path, delimiter='\t')
        vibration = data[:, 0]  

        # RMS 
        rms = np.sqrt(np.mean(vibration ** 2))
        rms_list.append(rms)

        # Spectrogram + Laplacian variance 
        f, t, Sxx = signal.spectrogram(vibration, fs=20000, nperseg=256, noverlap=128)
        Sxx_db = 10 * np.log10(Sxx + 1e-10)
        Sxx_norm = cv2.normalize(Sxx_db, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        img = cv2.applyColorMap(Sxx_norm, cv2.COLORMAP_JET)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        variance_list.append(lap_var)

        file_names.append(filename)
        print(f"Processed {filename}: RMS={rms:.4f}, Var={lap_var:.1f}")

    except Exception as e:
        print(f"Skipped {filename}: {e}")

print(f"\nTotal processed: {len(file_names)}")

# Saving to a CSV file
df.to_csv('bearing_analysis_dataset.csv', index=False)
print("Saved to bearing_analysis_dataset.csv")

In [ ]:
# comparison summary of RMS and Laplacian with histogram and control chart for 100 files

import numpy as np
import os
import matplotlib.pyplot as plt
import cv2
from scipy import signal
from scipy.stats import norm


# === Metrics ===
mean_rms = np.mean(rms_list)
std_rms = np.std(rms_list)
mean_var = np.mean(variance_list)
std_var = np.std(variance_list)

# Out-of-control counts ( 3-sigma rule)
ucl_rms = mean_rms + 3 * std_rms
ucl_var = mean_var + 3 * std_var

out_of_control_rms = sum(1 for x in rms_list if x > ucl_rms)
out_of_control_var = sum(1 for x in variance_list if x > ucl_var)

# Rough sigma levels (short-term, with 1.5 shift)
def sigma_level(defect_rate):
    if defect_rate == 0: return 6.0
    z = norm.ppf(1 - defect_rate)
    return z + 1.5

defect_rate_rms = out_of_control_rms / len(rms_list) if rms_list else 0
defect_rate_var = out_of_control_var / len(variance_list) if variance_list else 0

sigma_rms = sigma_level(defect_rate_rms)
sigma_var = sigma_level(defect_rate_var)


# === Plots ===

# 1. Histogram comparison 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(rms_list, bins=25, color='skyblue', edgecolor='black')
ax1.set_title("Histogram - RMS (Traditional)")
ax1.set_xlabel("RMS (g)")
ax1.set_ylabel("Count")
ax2.hist(variance_list, bins=25, color='salmon', edgecolor='black')
ax2.set_title("Histogram - Laplacian Variance (AI/OpenCV)")
ax2.set_xlabel("Variance")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

# 2. Control chart - RMS
plt.figure(figsize=(10, 5))
plt.plot(range(len(rms_list)), rms_list, 'b.-', label='RMS')
plt.axhline(mean_rms, color='green', linestyle='-', label='Mean')
plt.axhline(ucl_rms, color='red', linestyle='--', label='UCL')
plt.title("Control Chart - RMS (Traditional)")
plt.xlabel("Snapshot Number")
plt.ylabel("RMS (g)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 3. Control chart - Laplacian variance
plt.figure(figsize=(10, 5))
plt.plot(range(len(variance_list)), variance_list, 'm.-', label='Laplacian Variance')
plt.axhline(mean_var, color='green', linestyle='-', label='Mean')
plt.axhline(ucl_var, color='red', linestyle='--', label='UCL')
plt.title("Control Chart - Laplacian Variance (AI/OpenCV)")
plt.xlabel("Snapshot Number")
plt.ylabel("Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Traditional Scatter plot demonstration (RMS vs snapshot number) 

import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

corr, p_value = stats.pearsonr(df['snapshot'], df['rms'])

print(f"Pearson correlation between snapshot number and RMS: {corr:.3f}")
print(f"p-value: {p_value:.4f}")

# scatter plot to visualize
plt.figure(figsize=(7,4))
plt.scatter(df['snapshot'], df['rms'], alpha=0.6)
plt.xlabel('Snapshot Number (time)')
plt.ylabel('RMS (g)')
plt.title('RMS vs Time – Weak trend')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Boxplot of variance by condition group

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
sns.boxplot(x='condition', y='laplacian_variance', data=df, palette='Set2')
plt.title('Laplacian Variance by Bearing Condition')
plt.ylabel('Variance')
plt.xlabel('Condition')
plt.show()

In [ ]:
# ANOVA, K-Means, and scatter plot comparison 

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Traditional ANOVA on Laplacian variance across the 3 groups

print("=== ANOVA on Laplacian variance ===")

groups_var = [
    df[df['condition'] == 'Healthy']['laplacian_variance'],
    df[df['condition'] == 'Degraded']['laplacian_variance'],
    df[df['condition'] == 'Failing']['laplacian_variance'] ]

f_stat_var, p_val_var = stats.f_oneway(*groups_var)

print(f"F-statistic: {f_stat_var:.2f}")
print(f"p-value:     {p_val_var:.4e}")

if p_val_var < 0.05:
    print("→ Significant difference in means across conditions")
else:
    print("→ No significant difference detected")

# Showing group means and overlapping
print("\nGroup means (Laplacian variance):")
print(df.groupby('condition')['laplacian_variance'].agg(['mean', 'std', 'count']))


# 2. Traditional ANOVA on RMS (for comparison)

print("\n=== ANOVA on RMS ===")

groups_rms = [
    df[df['condition'] == 'Healthy']['rms'],
    df[df['condition'] == 'Degraded']['rms'],
    df[df['condition'] == 'Failing']['rms'] ]

f_stat_rms, p_val_rms = stats.f_oneway(*groups_rms)

print(f"F-statistic: {f_stat_rms:.2f}")
print(f"p-value:     {p_val_rms:.4e}")


# 3. Scikit-learn K-Means clustering (3 clusters)

print("\n=== K-Means Clustering (k=3) ===")

X = df[['rms', 'laplacian_variance']]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

# Showing how clusters map conditions
crosstab = pd.crosstab(df['condition'], df['cluster'], margins=True)
print("Cluster vs Condition crosstab:")
print(crosstab)

# Show cluster centroids (average RMS & variance per cluster)
centroids = pd.DataFrame(kmeans.cluster_centers_, columns=['rms', 'laplacian_variance'])
print("\nCluster centroids:")
print(centroids)


# 4. Visual: Scatter plot of clusters colored by condition

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x='rms',
    y='laplacian_variance',
    hue='cluster',
    style='condition',
    palette='deep',
    s=90,
    alpha=0.8 )

plt.title('K-Means Clusters (3) on RMS and Laplacian Variance')
plt.xlabel('RMS (g)')
plt.ylabel('Laplacian Variance')
plt.legend(title='Cluster / Condition', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Generating a bar chart of defect severity. ( number related to the saved csv code )

import matplotlib.pyplot as plt
import pandas as pd

# Create severity buckets from variance
df['severity'] = pd.cut(
    df['laplacian_variance'],
    bins=[-np.inf, 16500, 17200, np.inf],
    labels=['Degraded', 'Failing', 'Healthy'])

# Counting per bucket
counts = df['severity'].value_counts().sort_values(ascending=False)

# creating the Bar Chart of the defect severity
plt.figure(figsize=(8, 5))
counts.plot(kind='bar', color=['red', 'lightgreen', 'orange'])
plt.title('Distribution of Snapshots by Defect Severity (AI Method)')
plt.xlabel('Severity Bucket')
plt.ylabel('Number of Snapshots')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.show()

print(counts)

In [ ]:
# Generating a Pareto chart 

import matplotlib.pyplot as plt
import pandas as pd

# Creating severity buckets from variance
df['severity'] = pd.cut(df['laplacian_variance'], bins=[-np.inf, 16500, 17200, np.inf], labels=['Degraded', 'Failing', 'Healthy'])

# Counting per bucket
counts = df['severity'].value_counts().sort_values(ascending=False)

# Preparing data for Pareto Chart
categories = counts.index
values = counts.values

cumsum = np.cumsum(values)
total = values.sum()
cum_percentage = cumsum / total * 100

# Creating the figure
fig, ax1 = plt.subplots(figsize=(9, 5))

# Bar plot – frequencies
bars = ax1.bar(categories, values, color='skyblue', edgecolor='black')
ax1.set_xlabel('Condition Group')
ax1.set_ylabel('Number of Snapshots', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Secondary axis – cumulative percentage
ax2 = ax1.twinx()
ax2.plot(categories, cum_percentage, color='red', marker='o', linewidth=2, label='Cumulative %')
ax2.set_ylabel('Cumulative Percentage (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 110)

# Adding 80% line
ax2.axhline(y=80, color='gray', linestyle='--', alpha=0.7, label='80% line')

plt.title('Pareto Chart – Contribution of Condition Groups to Total Snapshots')
fig.tight_layout()
plt.show()

# displaying number of snapshots with cumulative percentage
print("Pareto data:")
for cat, val, perc in zip(categories, values, cum_percentage):
    print(f"{cat}: {val} snapshots ({perc:.1f}% cumulative)")

In [ ]:
# K-Means Scatter plot 

from sklearn.cluster import KMeans
import seaborn as sns

X = df[['rms', 'laplacian_variance']]  

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

# Cross-tabing to see how clusters match 
print(pd.crosstab(df['condition'], df['cluster'], margins=True))

# Visual: scatter of clusters
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='rms', y='laplacian_variance', hue='cluster', palette='deep', style='condition')
plt.title('K-Means Clusters (3) on RMS + Laplacian Variance')
plt.xlabel('RMS (g)')
plt.ylabel('Laplacian Variance')
plt.legend(title='Cluster / Condition')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# control chart of RMS

import matplotlib.pyplot as plt

mean_rms = np.mean(rms_list)
std_rms = np.std(rms_list)
ucl_rms = mean_rms + 3 * std_rms

plt.figure(figsize=(10, 5))
plt.plot(range(len(rms_list)), rms_list, 'b.-', label='RMS')
plt.axhline(mean_rms, color='green', label='Mean')
plt.axhline(ucl_rms, color='red', linestyle='--', label='UCL')
plt.title('Control Chart - RMS (Traditional)')
plt.xlabel('Snapshot Number')
plt.ylabel('RMS (g)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Out-of-control points (RMS > UCL):", sum(1 for x in rms_list if x > ucl_rms))

In [ ]:
# control chart of Laplacian variance

mean_var = np.mean(variance_list)
std_var = np.std(variance_list)
ucl_var = mean_var + 3 * std_var

plt.figure(figsize=(10, 5))
plt.plot(range(len(variance_list)), variance_list, 'm.-', label='Laplacian Variance')
plt.axhline(mean_var, color='green', label='Mean')
plt.axhline(ucl_var, color='red', linestyle='--', label='UCL')
plt.title('Control Chart - Laplacian Variance (AI/OpenCV)')
plt.xlabel('Snapshot Number')
plt.ylabel('Variance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Out-of-control points (Variance > UCL):", sum(1 for x in variance_list if x > ucl_var))

In [ ]:
#  histogram of the variance and RMS. 

from scipy import signal
import cv2

variance_list = []

for i, filename in enumerate(file_names):
    file_path = os.path.join(folder, filename)
    data = np.loadtxt(file_path, delimiter='\t')
    vibration = data[:, 0]
    
    f, t, Sxx = signal.spectrogram(vibration, fs=20000, nperseg=256, noverlap=128)
    Sxx_db = 10 * np.log10(Sxx + 1e-10)
    Sxx_norm = cv2.normalize(Sxx_db, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img = cv2.applyColorMap(Sxx_norm, cv2.COLORMAP_JET)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    variance_list.append(lap_var)
    
    print(f"{filename}: Var = {lap_var:.1f}")

# Histogram plot
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.hist(rms_list, bins=20, color='skyblue', edgecolor='black')
ax1.set_title('Histogram - RMS (Traditional)')
ax1.set_xlabel('RMS (g)')
ax1.set_ylabel('Count')

ax2.hist(variance_list, bins=20, color='salmon', edgecolor='black')
ax2.set_title('Histogram - Laplacian Variance (AI)')
ax2.set_xlabel('Variance')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Traditional Pilot Testing

import pandas as pd
import numpy as np   


# Selecting pilot subset
pilot_start = 30
pilot_end = pilot_start + 15                    
pilot_df = df.iloc[pilot_start:pilot_end].copy()


if pilot_df.empty:
    print("WARNING: pilot_df is empty! Using first 20 rows instead.")
    pilot_df = df.iloc[0:20].copy()

# Setting bearing conditions
pilot_df['condition'] = pd.cut(
    pilot_df['laplacian_variance'], 
    bins=[-np.inf, 16500, 17200, np.inf], 
    labels=['Healthy', 'Degraded', 'Failing'])

# Simulating the alert rule: two consecutive > 16,500
pilot_df['alert status'] = False
high_count = 0

for idx in pilot_df.index:
    if pilot_df.loc[idx, 'laplacian_variance'] > 16500:
        high_count += 1
        if high_count >= 2:
            pilot_df.loc[idx, 'alert status'] = True
            high_count = 0
    else:
        high_count = 0

# Adding notes column
pilot_df['notes'] = ''

for idx in pilot_df.index:
    if pilot_df.loc[idx, 'alert status']:
        pilot_df.loc[idx, 'notes'] = 'Alert triggered - two consecutive highs'
    elif pilot_df.loc[idx, 'laplacian_variance'] > 16500:
        pilot_df.loc[idx, 'notes'] = 'Monitor closely - Single high'
    else:
        pilot_df.loc[idx, 'notes'] = 'Good condition'

# Converting True/False to Yes/No
pilot_df['alert status'] = pilot_df['alert status'].map({True: 'Yes', False: 'No'})

# Final Table
pilot_table = pilot_df[['snapshot', 'rms', 'laplacian_variance', 'condition', 'alert status', 'notes']]

# Display
print("\nPilot Testing Snapshot Table")
display(pilot_table)

In [ ]:
# Traditional SPC Control chart

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("bearing_analysis_dataset.csv")

mean_rms = df['rms'].mean()
std_rms = df['rms'].std()

UCL = mean_rms + 3 * std_rms
LCL = mean_rms - 3 * std_rms

plt.figure(figsize=(10,5))
plt.plot(df['snapshot'], df['rms'], label="RMS")
plt.axhline(mean_rms, color='green', linestyle='--', label="Mean")
plt.axhline(UCL, color='red', linestyle='--', label="UCL")
plt.axhline(LCL, color='red', linestyle='--', label="LCL")

plt.title("Traditional SPC Control Chart (RMS)")
plt.xlabel("Snapshot")
plt.ylabel("RMS")
plt.legend()
plt.show()